# LeetCode 348 — Design Tic-Tac-Toe bằng Python

Link đề: https://leetcode.com/problems/design-tic-tac-toe/ · Độ khó: **Medium** · Chủ đề: Design (thiết kế lớp), Array, Hash Table, Matrix (ma trận), Simulation (mô phỏng), Counting (đếm)

Skeleton LeetCode:

```python
class TicTacToe:

    def __init__(self, n: int):
        ...

    def move(self, row: int, col: int, player: int) -> int:
        ...


# Your TicTacToe object will be instantiated and called as such:
# obj = TicTacToe(n)
# param_1 = obj.move(row, col, player)
```

Ghi chú: đây là bài **Premium**; đề được viết theo hiểu biết về bài (ví dụ 1 lấy theo mô tả phổ biến của bài), nên **đối chiếu lại với đề gốc nếu bạn có tài khoản Premium**. Mọi đáp án trong notebook đã được kiểm chứng bằng code (so với một bộ kiểm tra độc lập quét cả bàn cờ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế lớp** `TicTacToe`: hàm khởi tạo `__init__(n)` dựng bàn cờ `n × n`, và `move(row, col, player)` ghi một nước đi rồi cho biết **ai thắng** (`0` nếu chưa ai thắng, `1` hoặc `2` nếu người đó thắng).
2. Nắm các công cụ Python cho bài thiết kế: `class`, `self` và **trạng thái giữ lại giữa các lần gọi**, danh sách hai chiều `[[0] * n for _ in range(n)]`, `all` / `any`, `abs`, và **bitmask** (`1 << c`, `|`, `==`).
3. Đi từ **lưu bàn cờ rồi quét cả bàn mỗi nước `O(n²)`** → **chỉ quét 4 đường qua ô vừa đánh `O(n)`** → **chỉ lưu bộ đếm +1/−1 cho hàng, cột, hai đường chéo `O(1)`** → **bitmask cho từng người `O(1)`**.
4. Hiểu mấu chốt: để biết ai thắng **không cần nhớ cả bàn cờ**, chỉ cần biết trên mỗi hàng/cột/đường chéo **mỗi người đã có bao nhiêu quân**; người 1 đếm `+1`, người 2 đếm `−1`, ai đạt `±n` là thắng.
5. Tự dựng **bộ chạy chuỗi lệnh** mô phỏng cách LeetCode gọi lớp (danh sách tên hàm + danh sách tham số) và kiểm thử bằng **ngẫu nhiên có seed** so với oracle độc lập.

## 2. Đề bài & ví dụ

Giả sử có luật chơi cờ ca-rô (tic-tac-toe) trên bàn `n × n` giữa hai người chơi:

1. Mỗi nước đi **được đảm bảo hợp lệ** và đặt vào **ô trống**.
2. Khi đã có người thắng thì **không còn nước đi** nào nữa.
3. Người đầu tiên có **`n` quân của mình** trên cùng một **hàng**, **cột**, hoặc **đường chéo** (chính hay phụ) là người thắng.

Hãy cài đặt lớp `TicTacToe`:

- `TicTacToe(int n)`: khởi tạo bàn cờ `n × n`.
- `int move(int row, int col, int player)`: người chơi `player` đánh vào ô `(row, col)`. Nước đi **hợp lệ**, và hai người **luân phiên** đánh. Trả về:
  - `0` nếu **chưa có người thắng** sau nước đi,
  - `1` nếu **người chơi 1** thắng,
  - `2` nếu **người chơi 2** thắng.

**Ví dụ 1** (bàn `3 × 3`, người 1 là `X`, người 2 là `O`)

```text
Input
["TicTacToe", "move", "move", "move", "move", "move", "move", "move"]
[[3], [0, 0, 1], [0, 2, 2], [2, 2, 1], [1, 1, 2], [2, 0, 1], [1, 0, 2], [2, 1, 1]]
Output
[null, 0, 0, 0, 0, 0, 0, 1]
```

Sau nước cuối cùng, hàng `2` là `X X X`: người 1 thắng.

**Ràng buộc:**

- `2 <= n <= 100`
- `player` là `1` hoặc `2`
- `0 <= row, col < n`
- `(row, col)` là **duy nhất** với mỗi lần gọi `move`
- Có tối đa `n²` lần gọi `move`

**Follow-up:** bạn có thể làm tốt hơn `O(n²)` **mỗi lần `move()`** không?

In [1]:
ops = ["TicTacToe", "move", "move", "move", "move", "move", "move", "move"]
args = [[3], [0, 0, 1], [0, 2, 2], [2, 2, 1], [1, 1, 2], [2, 0, 1], [1, 0, 2], [2, 1, 1]]
expected = [None, 0, 0, 0, 0, 0, 0, 1]

print(list(zip(ops, args)))
print("kỳ vọng:", expected)

[('TicTacToe', [3]), ('move', [0, 0, 1]), ('move', [0, 2, 2]), ('move', [2, 2, 1]), ('move', [1, 1, 2]), ('move', [2, 0, 1]), ('move', [1, 0, 2]), ('move', [2, 1, 1])]
kỳ vọng: [None, 0, 0, 0, 0, 0, 0, 1]


## 3. Kiến thức nền

### 3.1. Lớp (`class`), `self` và trạng thái giữ lại giữa các lần gọi

Ở các bài thiết kế, ta không viết một hàm mà viết một **lớp**. `__init__(self, ...)` chạy **một lần** khi tạo đối tượng và thường gán các thuộc tính `self.xxx = ...` — đây là **trạng thái** của đối tượng. Các hàm khác (như `move`) **dùng và cập nhật** `self.xxx`, nên kết quả của lần gọi sau phụ thuộc các lần gọi trước. Đó chính là ý nghĩa của "thiết kế": ta quyết định **lưu gì** để các hàm trả lời nhanh.

In [2]:
class Counter3:
    def __init__(self, start):
        self.value = start                 # trạng thái được giữ giữa các lần gọi

    def add(self, delta):
        self.value += delta
        return self.value


c = Counter3(10)
print(c.add(1), c.add(5), c.add(-2))       # 11 16 14
d = Counter3(0)                            # đối tượng khác có trạng thái RIÊNG
print(d.add(1), c.value)                   # 1 14

11 16 14
1 14


### 3.2. Cách LeetCode gọi lớp: danh sách tên lệnh + danh sách tham số

Với bài thiết kế, ví dụ ghi hai danh sách song song: `ops` (tên lệnh; phần tử đầu là **tên lớp** tức là gọi hàm khởi tạo) và `args` (tham số cho từng lệnh). Kết quả là danh sách đầu ra, hàm khởi tạo cho `null` (`None`). Ta tự viết bộ chạy `run_commands` với `getattr(obj, tên)` để gọi hàm theo tên chuỗi.

In [3]:
def run_commands(cls, ops, args):
    """Mô phỏng cách LeetCode gọi: ops[0] là tên lớp (khởi tạo), các lệnh sau là tên phương thức."""
    obj = None
    out = []
    for name, a in zip(ops, args):
        if obj is None:
            obj = cls(*a)                       # lệnh đầu tiên: gọi hàm khởi tạo
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))  # gọi phương thức theo tên chuỗi
    return out


print(run_commands(Counter3, ["Counter3", "add", "add", "add"], [[10], [1], [5], [-2]]))
# [None, 11, 16, 14]

[None, 11, 16, 14]


### 3.3. Ma trận `n × n`, hàng, cột và hai đường chéo

Bàn cờ là **list của các list**: `board[r][c]` là ô ở hàng `r`, cột `c`. **Cẩn thận** khi tạo: phải dùng `[[0] * n for _ in range(n)]` (mỗi hàng là list **riêng**); `[[0] * n] * n` làm các hàng **trùng nhau**.

Với ô `(r, c)` trên bàn `n × n`:

- Nó thuộc **hàng `r`** và **cột `c`** (luôn luôn).
- Nó thuộc **đường chéo chính** khi `r == c`.
- Nó thuộc **đường chéo phụ** khi `r + c == n - 1`.

Vậy mỗi ô thuộc **tối đa 4** đường, và **tổng cộng chỉ có `2n + 2` đường** cần để ý.

In [4]:
n = 3
good = [[0] * n for _ in range(n)]
good[0][0] = 1
print(good)                                 # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
bad = [[0] * n] * n
bad[0][0] = 1
print(bad)                                  # [[1, 0, 0], [1, 0, 0], [1, 0, 0]]  <- ba hàng là MỘT list

for r, c in [(0, 0), (1, 1), (0, 2), (1, 2), (2, 2)]:
    print((r, c), "chéo chính" if r == c else "", "chéo phụ" if r + c == n - 1 else "")

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]
[[1, 0, 0], [1, 0, 0], [1, 0, 0]]
(0, 0) chéo chính 
(1, 1) chéo chính chéo phụ
(0, 2)  chéo phụ
(1, 2)  
(2, 2) chéo chính 


### 3.4. `all`, `any`, `abs`

- `all(biểu_thức for ...)` là `True` khi **mọi** phần tử thỏa; `any(...)` khi **có ít nhất một**.
- `abs(x)` là giá trị tuyệt đối: `abs(-3) == 3`. Ta sẽ dùng khi bộ đếm là `+n` hoặc `−n`.

In [5]:
row = [1, 1, 1]
print(all(x == 1 for x in row), all(x == 1 for x in [1, 0, 1]))     # True False
print(any(x == 2 for x in row), any(x == 0 for x in [1, 0, 1]))     # False True
print(abs(-3), abs(3))                                               # 3 3

True False
False True
3 3


### 3.5. Bitmask: một tập vị trí lưu trong một số nguyên

Ta có thể dùng **bit** của một số nguyên làm "tập các vị trí": bit `c` bằng `1` nghĩa là "ô thứ `c` đã có quân". `1 << c` là số có đúng bit `c` bằng `1`; `mask |= 1 << c` **bật** bit `c`; `(1 << n) - 1` là số có `n` bit `1` liên tiếp (tất cả các bit từ `0` đến `n - 1`). Một hàng đã **đầy** khi `mask == (1 << n) - 1`.

In [6]:
n = 3
full = (1 << n) - 1
print(1 << 0, 1 << 1, 1 << 2, full, bin(full))        # 1 2 4 7 0b111
mask = 0
for c in [0, 2]:
    mask |= 1 << c                                    # bật bit c
print(mask, bin(mask), mask == full)                  # 5 0b101 False
mask |= 1 << 1
print(mask, bin(mask), mask == full)                  # 7 0b111 True

1 2 4 7 0b111
5 0b101 False
7 0b111 True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy đi lại **Ví dụ 1** bằng tay. Bàn `3 × 3`, kí hiệu `.` là ô trống:

| Nước | Người | Ô | Bàn cờ sau nước đi | Thắng? |
|---|---|---|---|---|
| 1 | 1 (`X`) | (0,0) | `X . .` / `. . .` / `. . .` | không |
| 2 | 2 (`O`) | (0,2) | `X . O` / `. . .` / `. . .` | không |
| 3 | 1 (`X`) | (2,2) | `X . O` / `. . .` / `. . X` | không |
| 4 | 2 (`O`) | (1,1) | `X . O` / `. O .` / `. . X` | không |
| 5 | 1 (`X`) | (2,0) | `X . O` / `. O .` / `X . X` | không |
| 6 | 2 (`O`) | (1,0) | `X . O` / `O O .` / `X . X` | không |
| 7 | 1 (`X`) | (2,1) | `X . O` / `O O .` / `X X X` | **người 1** |

Hai nhận xét làm nên lời giải nhanh:

1. **Chỉ ô vừa đánh mới có thể tạo ra chiến thắng mới.** Trước nước này chưa ai thắng, nên chỉ cần xem **các đường đi qua ô đó** (tối đa 4 đường).
2. Để biết một đường đã đủ `n` quân của cùng một người, **không cần nhớ từng ô** — chỉ cần **đếm** số quân của mỗi người trên đường đó.

In [7]:
# vẽ lại bàn cờ ví dụ 1 để "thấy" các đường
def render(board):
    sym = {0: ".", 1: "X", 2: "O"}
    return "\n".join(" ".join(sym[v] for v in row) for row in board)


board = [[0] * 3 for _ in range(3)]
for r, c, p in [(0, 0, 1), (0, 2, 2), (2, 2, 1), (1, 1, 2), (2, 0, 1), (1, 0, 2), (2, 1, 1)]:
    board[r][c] = p
print(render(board))
print("hàng 2:", board[2], "-> người thắng", set(board[2]))

X . O
O O .
X X X
hàng 2: [1, 1, 1] -> người thắng {1}


## 5. Cách 1 — Brute Force / baseline: lưu bàn cờ và quét cả bàn mỗi nước

**Ý tưởng.** Lưu `board[r][c] = người đã đánh` (`0` là trống). Sau mỗi nước đi, **quét toàn bộ** `n` hàng, `n` cột và hai đường chéo, xem có đường nào toàn quân của cùng một người không. Rõ ràng đúng nhưng chậm.

In [8]:
class TicTacToeScanAll:
    def __init__(self, n: int):
        self.n = n
        self.board = [[0] * n for _ in range(n)]

    def move(self, row: int, col: int, player: int) -> int:
        n, b = self.n, self.board
        b[row][col] = player
        lines = []
        for i in range(n):
            lines.append([b[i][j] for j in range(n)])            # hàng i
            lines.append([b[j][i] for j in range(n)])            # cột i
        lines.append([b[i][i] for i in range(n)])               # chéo chính
        lines.append([b[i][n - 1 - i] for i in range(n)])       # chéo phụ
        for line in lines:
            if line[0] != 0 and all(x == line[0] for x in line):
                return line[0]                                   # đường toàn một người
        return 0


print(run_commands(TicTacToeScanAll, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )


**Độ phức tạp (Cách 1).** Mỗi `move` quét `2n + 2` đường, mỗi đường `n` ô ⇒ **Time `O(n²)` mỗi nước**; Space `O(n²)` cho bàn cờ. Với `n = 100` và tối đa `10⁴` nước, tổng cỡ `10⁸` phép — quá chậm trong Python.

**Nút thắt:** (1) ta quét cả những đường **không liên quan** tới ô vừa đánh; (2) ta duyệt `n` ô của mỗi đường để **đếm lại** những gì đã biết từ trước. Hãy chỉ xét đường qua ô vừa đánh, rồi tìm cách tránh đếm lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Bộ đếm.** Giữ các bộ đếm thay vì cả bàn cờ:

- `rows[r]`, `cols[c]` (mỗi cái có `n` ô), `diag` (đường chéo chính), `anti` (đường chéo phụ).

Khi người `p` đánh vào `(r, c)`, cộng **`+1` nếu `p == 1`**, **`−1` nếu `p == 2`** vào `rows[r]`, `cols[c]`, thêm `diag` nếu `r == c`, thêm `anti` nếu `r + c == n - 1`. Một đường đã có **`n` quân của người 1** khi bộ đếm của nó bằng `+n`; **`n` quân của người 2** khi bằng `−n`. Hai người không cùng ở một đường mà cho ra `±n` vì quân của hai người **triệt tiêu nhau** (một đường chứa cả `X` lẫn `O` có `|tổng| < n`).

Mỗi `move` chỉ cập nhật tối đa **4 con số** và kiểm tra tối đa 4 con số ⇒ **`O(1)`**, và bộ nhớ chỉ `O(n)`.

Tính tay `n = 3` cho 3 nước đầu của ví dụ: `(0,0,1)`: `rows[0] = 1`, `cols[0] = 1`, `diag = 1`. `(0,2,2)`: `rows[0] = 0`, `cols[2] = −1`, `anti = −1`. `(2,2,1)`: `rows[2] = 1`, `cols[2] = 0`, `diag = 2`.

In [9]:
n = 3
rows, cols = [0] * n, [0] * n
diag = anti = 0
for r, c, p in [(0, 0, 1), (0, 2, 2), (2, 2, 1)]:
    d = 1 if p == 1 else -1
    rows[r] += d
    cols[c] += d
    if r == c:
        diag += d
    if r + c == n - 1:
        anti += d
    print((r, c, p), "rows", rows, "cols", cols, "diag", diag, "anti", anti)

(0, 0, 1) rows [1, 0, 0] cols [1, 0, 0] diag 1 anti 0
(0, 2, 2) rows [0, 0, 0] cols [1, 0, -1] diag 1 anti -1
(2, 2, 1) rows [0, 0, 1] cols [1, 0, 0] diag 2 anti -1


## 7. Các cách tiếp theo

### Cách 2 — Lưu bàn cờ, chỉ quét **4 đường qua ô vừa đánh**

**Giải thích từng đoạn.**

1. Vẫn lưu `board`, đặt `board[row][col] = player`.
2. Kiểm tra **hàng `row`**: mọi ô có bằng `player` không?
3. Kiểm tra **cột `col`**.
4. Nếu `row == col`: kiểm tra **đường chéo chính**; nếu `row + col == n - 1`: kiểm tra **đường chéo phụ**.
5. Thấy đường đủ `n` quân của `player` thì trả `player`, ngược lại `0`.

In [10]:
class TicTacToeLinesDemo:
    def __init__(self, n: int):
        self.n = n
        self.board = [[0] * n for _ in range(n)]

    def move(self, row: int, col: int, player: int) -> int:
        n, b = self.n, self.board
        b[row][col] = player
        if all(b[row][j] == player for j in range(n)):                       # hàng của ô
            return player
        if all(b[i][col] == player for i in range(n)):                       # cột của ô
            return player
        if row == col and all(b[i][i] == player for i in range(n)):          # chéo chính
            return player
        if row + col == n - 1 and all(b[i][n - 1 - i] == player for i in range(n)):   # chéo phụ
            return player
        return 0


print(run_commands(TicTacToeLinesDemo, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành TicTacToe khi nộp)
class TicTacToeLines:
    def __init__(self, n: int):
        self.n = n
        self.board = [[0] * n for _ in range(n)]

    def move(self, row: int, col: int, player: int) -> int:
        n, b = self.n, self.board
        b[row][col] = player
        if (all(b[row][j] == player for j in range(n))
                or all(b[i][col] == player for i in range(n))
                or (row == col and all(b[i][i] == player for i in range(n)))
                or (row + col == n - 1 and all(b[i][n - 1 - i] == player for i in range(n)))):
            return player
        return 0


print(run_commands(TicTacToeLines, ops, args))

[None, 0, 0, 0, 0, 0, 0, 1]


**Độ phức tạp Cách 2:** mỗi `move` kiểm tra tối đa 4 đường, mỗi đường `n` ô ⇒ **Time `O(n)` mỗi nước**; **Space `O(n²)`** cho bàn cờ.

### Cách 3 — Bộ đếm `+1/−1` cho hàng, cột, hai đường chéo (`O(1)` mỗi nước)

**Giải thích từng đoạn.**

1. `self.rows = [0] * n`, `self.cols = [0] * n`, `self.diag = 0`, `self.anti = 0`.
2. `delta = 1 if player == 1 else -1`.
3. `self.rows[row] += delta`, `self.cols[col] += delta`; nếu `row == col`: `self.diag += delta`; nếu `row + col == n - 1`: `self.anti += delta`.
4. Nếu **một trong các bộ đếm vừa cập nhật** có `abs(...) == n`, trả `player`; ngược lại `0`.

In [12]:
class TicTacToeCounterDemo:
    def __init__(self, n: int):
        self.n = n
        self.rows = [0] * n
        self.cols = [0] * n
        self.diag = 0
        self.anti = 0

    def move(self, row: int, col: int, player: int) -> int:
        delta = 1 if player == 1 else -1            # người 1: +1, người 2: -1
        self.rows[row] += delta
        self.cols[col] += delta
        if row == col:
            self.diag += delta
        if row + col == self.n - 1:
            self.anti += delta
        n = self.n
        if abs(self.rows[row]) == n or abs(self.cols[col]) == n \
                or abs(self.diag) == n or abs(self.anti) == n:
            return player
        return 0


print(run_commands(TicTacToeCounterDemo, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )


In [13]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành TicTacToe khi nộp)
class TicTacToeCounter:
    def __init__(self, n: int):
        self.n = n
        self.rows = [0] * n
        self.cols = [0] * n
        self.diag = 0
        self.anti = 0

    def move(self, row: int, col: int, player: int) -> int:
        delta = 1 if player == 1 else -1
        self.rows[row] += delta
        self.cols[col] += delta
        if row == col:
            self.diag += delta
        if row + col == self.n - 1:
            self.anti += delta
        n = self.n
        if n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag), abs(self.anti)):
            return player
        return 0


print(run_commands(TicTacToeCounter, ops, args))

[None, 0, 0, 0, 0, 0, 0, 1]


**Độ phức tạp Cách 3:** **Time `O(1)` mỗi nước**, **Space `O(n)`** (hai mảng `n` phần tử và hai số).

Ghi chú: ta chỉ kiểm tra các bộ đếm **vừa thay đổi** vì trước nước này chưa ai thắng; các bộ đếm khác không đổi nên không thể vừa chạm `±n`. (Nếu `diag` không thuộc ô vừa đánh thì `abs(diag) == n` đã sai từ trước, nên kiểm tra thêm cũng vô hại.)

### Cách 4 — Bitmask cho từng người chơi

**Ý tưởng.** Thay vì đếm, lưu **tập vị trí** bằng bit. Với mỗi người `p ∈ {1, 2}`:

- `rows[p][r]`: số nguyên có bit `c` bật nếu người `p` có quân ở `(r, c)`;
- `cols[p][c]`: bit `r` bật nếu người `p` có quân ở `(r, c)`;
- `diag[p]`: bit `r` bật nếu người `p` có quân ở `(r, r)`;
- `anti[p]`: bit `r` bật nếu người `p` có quân ở `(r, n - 1 - r)`.

Một đường **đủ `n` quân** của `p` khi số nguyên tương ứng bằng `full = (1 << n) - 1`. Khác với bộ đếm, cách này nhớ **vị trí chính xác**, hữu ích nếu sau này cần "ô nào còn thiếu" (ví dụ để làm AI chơi cờ).

In [14]:
class TicTacToeBitmaskDemo:
    def __init__(self, n: int):
        self.n = n
        self.full = (1 << n) - 1
        self.rows = {1: [0] * n, 2: [0] * n}
        self.cols = {1: [0] * n, 2: [0] * n}
        self.diag = {1: 0, 2: 0}
        self.anti = {1: 0, 2: 0}

    def move(self, row: int, col: int, player: int) -> int:
        n, full = self.n, self.full
        self.rows[player][row] |= 1 << col          # hàng `row`: bật bit `col`
        self.cols[player][col] |= 1 << row          # cột `col`: bật bit `row`
        if row == col:
            self.diag[player] |= 1 << row
        if row + col == n - 1:
            self.anti[player] |= 1 << row
        if (self.rows[player][row] == full or self.cols[player][col] == full
                or self.diag[player] == full or self.anti[player] == full):
            return player
        return 0


print(run_commands(TicTacToeBitmaskDemo, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )


In [15]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành TicTacToe khi nộp)
class TicTacToeBitmask:
    def __init__(self, n: int):
        self.n = n
        self.full = (1 << n) - 1
        self.rows = [[0] * n, [0] * n]               # rows[p - 1][r]
        self.cols = [[0] * n, [0] * n]
        self.diag = [0, 0]
        self.anti = [0, 0]

    def move(self, row: int, col: int, player: int) -> int:
        p = player - 1
        self.rows[p][row] |= 1 << col
        self.cols[p][col] |= 1 << row
        if row == col:
            self.diag[p] |= 1 << row
        if row + col == self.n - 1:
            self.anti[p] |= 1 << row
        full = self.full
        if full in (self.rows[p][row], self.cols[p][col], self.diag[p], self.anti[p]):
            return player
        return 0


print(run_commands(TicTacToeBitmask, ops, args))

[None, 0, 0, 0, 0, 0, 0, 1]


**Độ phức tạp Cách 4:** mỗi `move` làm vài phép `|` và so sánh trên số nguyên `n` bit (với `n <= 100`, một số nguyên Python nhỏ) ⇒ **Time `O(1)` mỗi nước** (coi phép trên số `n` bit là hằng; chặt chẽ là `O(n / w)`), **Space `O(n)`**.

## 8. TRACE từng bước

Ta trace Cách 3 (bộ đếm) trên **Ví dụ 1**, in các bộ đếm sau từng nước, rồi trace Cách 4 (bitmask) trên một ván `n = 3` ngắn hơn để thấy các bit.

In [16]:
def trace_counter(n, moves):
    rows, cols = [0] * n, [0] * n
    diag = anti = 0
    print(f"n = {n}")
    results = []
    for step, (r, c, p) in enumerate(moves, 1):
        d = 1 if p == 1 else -1
        rows[r] += d
        cols[c] += d
        if r == c:
            diag += d
        if r + c == n - 1:
            anti += d
        winner = p if n in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)) else 0
        print(f"  nước {step}: người {p} đánh ({r},{c}) delta={d:+d} | rows={rows} cols={cols} diag={diag} anti={anti} -> {winner}")
        results.append(winner)
    return results


moves1 = [(0, 0, 1), (0, 2, 2), (2, 2, 1), (1, 1, 2), (2, 0, 1), (1, 0, 2), (2, 1, 1)]
assert trace_counter(3, moves1) == [0, 0, 0, 0, 0, 0, 1]

n = 3
  nước 1: người 1 đánh (0,0) delta=+1 | rows=[1, 0, 0] cols=[1, 0, 0] diag=1 anti=0 -> 0
  nước 2: người 2 đánh (0,2) delta=-1 | rows=[0, 0, 0] cols=[1, 0, -1] diag=1 anti=-1 -> 0
  nước 3: người 1 đánh (2,2) delta=+1 | rows=[0, 0, 1] cols=[1, 0, 0] diag=2 anti=-1 -> 0
  nước 4: người 2 đánh (1,1) delta=-1 | rows=[0, -1, 1] cols=[1, -1, 0] diag=1 anti=-2 -> 0
  nước 5: người 1 đánh (2,0) delta=+1 | rows=[0, -1, 2] cols=[2, -1, 0] diag=1 anti=-1 -> 0
  nước 6: người 2 đánh (1,0) delta=-1 | rows=[0, -2, 2] cols=[1, -1, 0] diag=1 anti=-1 -> 0
  nước 7: người 1 đánh (2,1) delta=+1 | rows=[0, -2, 3] cols=[1, 0, 0] diag=1 anti=-1 -> 1


In [17]:
def trace_bitmask(n, moves):
    full = (1 << n) - 1
    rows = {1: [0] * n, 2: [0] * n}
    cols = {1: [0] * n, 2: [0] * n}
    diag, anti = {1: 0, 2: 0}, {1: 0, 2: 0}
    print(f"n = {n}, full = {full} = {bin(full)}")
    results = []
    for step, (r, c, p) in enumerate(moves, 1):
        rows[p][r] |= 1 << c
        cols[p][c] |= 1 << r
        if r == c:
            diag[p] |= 1 << r
        if r + c == n - 1:
            anti[p] |= 1 << r
        winner = p if full in (rows[p][r], cols[p][c], diag[p], anti[p]) else 0
        print(f"  nước {step}: người {p} đánh ({r},{c}) | hàng {r}: {rows[p][r]:0{n}b}, cột {c}: {cols[p][c]:0{n}b}, "
              f"chéo: {diag[p]:0{n}b}, chéo phụ: {anti[p]:0{n}b} -> {winner}")
        results.append(winner)
    return results


moves2 = [(1, 1, 1), (0, 1, 2), (0, 0, 1), (2, 2, 2), (2, 0, 1), (1, 0, 2), (0, 2, 1)]
assert trace_bitmask(3, moves2) == [0, 0, 0, 0, 0, 0, 1]

n = 3, full = 7 = 0b111
  nước 1: người 1 đánh (1,1) | hàng 1: 010, cột 1: 010, chéo: 010, chéo phụ: 010 -> 0
  nước 2: người 2 đánh (0,1) | hàng 0: 010, cột 1: 001, chéo: 000, chéo phụ: 000 -> 0
  nước 3: người 1 đánh (0,0) | hàng 0: 001, cột 0: 001, chéo: 011, chéo phụ: 010 -> 0
  nước 4: người 2 đánh (2,2) | hàng 2: 100, cột 2: 100, chéo: 100, chéo phụ: 000 -> 0
  nước 5: người 1 đánh (2,0) | hàng 2: 001, cột 0: 101, chéo: 011, chéo phụ: 110 -> 0
  nước 6: người 2 đánh (1,0) | hàng 1: 001, cột 0: 010, chéo: 100, chéo phụ: 000 -> 0
  nước 7: người 1 đánh (0,2) | hàng 0: 101, cột 2: 001, chéo: 011, chéo phụ: 111 -> 1


Trong ván thứ hai, người 1 chiếm `(1,1)`, `(0,0)`, `(2,0)` rồi cuối cùng `(0,2)`: đường chéo phụ gồm `(0,2)`, `(1,1)`, `(2,0)` nên ở nước cuối bitmask chéo phụ của người 1 thành `111` và **người 1 thắng**. Trước đó cột chéo phụ chỉ là `010` rồi `110`... (bit nào còn thiếu thì ô đó chưa có quân), nhờ vậy bitmask cho biết **ô nào còn thiếu**.

Bảng trace Cách 3 cho Ví dụ 1 (`n = 3`):

| nước | người, ô | `rows` | `cols` | `diag` | `anti` | kết quả |
|---|---|---|---|---|---|---|
| 1 | 1, (0,0) | `[1,0,0]` | `[1,0,0]` | 1 | 0 | 0 |
| 2 | 2, (0,2) | `[0,0,0]` | `[1,0,-1]` | 1 | -1 | 0 |
| 3 | 1, (2,2) | `[0,0,1]` | `[1,0,0]` | 2 | -1 | 0 |
| 4 | 2, (1,1) | `[0,-1,1]` | `[1,-1,0]` | 1 | -2 | 0 |
| 5 | 1, (2,0) | `[0,-1,2]` | `[2,-1,0]` | 1 | -1 | 0 |
| 6 | 2, (1,0) | `[0,-2,2]` | `[1,-1,0]` | 1 | -1 | 0 |
| 7 | 1, (2,1) | `[0,-2,3]` | `[1,0,0]` | 1 | -1 | **1** (vì `rows[2] = 3`) |

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3 (bộ đếm `+1/−1`)**: ngắn, `O(1)` mỗi nước, `O(n)` bộ nhớ, đáp ứng follow-up. Đây là bản đổi tên lớp thành `TicTacToe` đúng skeleton của đề.

In [18]:
class TicTacToe:

    def __init__(self, n: int):
        self.n = n
        self.rows = [0] * n          # rows[r]: (số quân người 1) - (số quân người 2) trên hàng r
        self.cols = [0] * n
        self.diag = 0                # đường chéo chính (row == col)
        self.anti = 0                # đường chéo phụ (row + col == n - 1)

    def move(self, row: int, col: int, player: int) -> int:
        delta = 1 if player == 1 else -1
        self.rows[row] += delta
        self.cols[col] += delta
        if row == col:
            self.diag += delta
        if row + col == self.n - 1:
            self.anti += delta
        n = self.n
        if n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag), abs(self.anti)):
            return player            # ai đạt +-n thì người đó thắng
        return 0


# Your TicTacToe object will be instantiated and called as such:
# obj = TicTacToe(n)
# param_1 = obj.move(row, col, player)

print(run_commands(TicTacToe, ops, args), "(kỳ vọng", expected, ")")

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )


## 10. Vì sao thuật toán đúng

**Ý nghĩa bộ đếm.** Với mỗi đường `L` (một hàng, một cột, hoặc một đường chéo), gọi `a` là số quân người 1 và `b` là số quân người 2 trên `L`. Bất biến: bộ đếm của `L` luôn bằng `a − b`.

- Mỗi nước đi của người 1 vào ô thuộc `L` làm `a` tăng 1, nên bộ đếm `+1`; người 2 làm `b` tăng 1, nên `−1`. Ta cập nhật đúng các đường mà ô `(r, c)` thuộc về: hàng `r`, cột `c`, đường chéo chính nếu `r == c`, đường chéo phụ nếu `r + c == n - 1`.
- `L` có `n` ô. Nếu bộ đếm bằng `+n` thì `a − b = n` mà `a + b <= n` và `b >= 0` ⇒ `a = n`, `b = 0`: **cả đường là quân người 1**. Tương tự `−n` là toàn quân người 2. Ngược lại nếu cả đường là quân một người thì bộ đếm bằng `±n`. Vậy **"đường đủ `n` quân của một người" ⇔ `|bộ đếm| = n`**.

**Chỉ cần kiểm tra bốn đường qua ô vừa đánh.** Trước nước đi này chưa ai thắng (đề: khi đã thắng thì không còn nước đi). Một đường **không chứa** ô vừa đánh giữ nguyên nội dung, nên không thể vừa mới đủ `n` quân. Vậy nếu có người thắng ngay sau nước này thì đường thắng phải qua ô `(row, col)`, và người thắng chính là `player` (chỉ quân của `player` vừa được thêm).

**Hai người không nhầm lẫn.** `X` và `O` cùng nằm trên một đường làm `|a − b| < n` (vì `a, b >= 1` và `a + b <= n` nên `|a − b| <= n − 2`), nên không tạo ra thắng giả.

**Cách 2 và Cách 1** đúng trực tiếp theo định nghĩa (quét đường); **Cách 4** đúng vì `rows[p][r] == full` ⇔ mọi cột `c` của hàng `r` đều là quân người `p`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là kích thước bàn cờ (`n <= 100`), có tối đa `n²` nước.

| Cách | Time mỗi `move` | Space | Tổng cho `n²` nước |
|---|---|---|---|
| 1. Quét cả bàn | `O(n²)` | `O(n²)` | `O(n⁴)` |
| 2. Quét 4 đường qua ô | `O(n)` | `O(n²)` | `O(n³)` |
| 3. Bộ đếm `+1/−1` | `O(1)` | `O(n)` | `O(n²)` |
| 4. Bitmask | `O(1)` (coi số `n` bit là hằng) | `O(n)` | `O(n²)` |

Phần khởi tạo: Cách 1–2 tốn `O(n²)` để tạo bàn; Cách 3–4 chỉ `O(n)`.

## 12. Bộ test

Ta kiểm tra bằng **bộ chạy chuỗi lệnh** `run_commands` và một **oracle độc lập** `oracle_winner(board)` quét toàn bàn (không dùng bộ đếm, không dùng bitmask). Mọi cách phải trả đúng kết quả sau **từng nước**.

In [19]:
import itertools
import random
import time

impls = {
    "scan_all": TicTacToeScanAll,
    "lines_demo": TicTacToeLinesDemo,
    "lines": TicTacToeLines,
    "counter_demo": TicTacToeCounterDemo,
    "counter": TicTacToeCounter,
    "bitmask_demo": TicTacToeBitmaskDemo,
    "bitmask": TicTacToeBitmask,
    "leetcode": TicTacToe,
}


def oracle_winner(board):
    """Oracle độc lập: liệt kê mọi đường rồi xem có đường nào toàn một người không."""
    n = len(board)
    lines = [[board[r][c] for c in range(n)] for r in range(n)]
    lines += [[board[r][c] for r in range(n)] for c in range(n)]
    lines.append([board[i][i] for i in range(n)])
    lines.append([board[i][n - 1 - i] for i in range(n)])
    for line in lines:
        if line[0] != 0 and len(set(line)) == 1:
            return line[0]
    return 0


def oracle_run(n, moves):
    board = [[0] * n for _ in range(n)]
    out = []
    for r, c, p in moves:
        assert board[r][c] == 0
        board[r][c] = p
        out.append(oracle_winner(board))
    return out


def check_game(n, moves, label):
    ops_g = ["TicTacToe"] + ["move"] * len(moves)
    args_g = [[n]] + [[r, c, p] for r, c, p in moves]
    ref = [None] + oracle_run(n, moves)
    for key, cls in impls.items():
        got = run_commands(cls, ops_g, args_g)
        assert got == ref, (label, key, n, moves, got, ref)
    return ref


for key, cls in impls.items():
    assert run_commands(cls, ops, args) == expected, key
print("PASS: ví dụ đề cho tất cả", len(impls), "cách")

PASS: ví dụ đề cho tất cả 8 cách


### 12.1. Các trường hợp biên

`n = 2` (hai quân đã đủ một đường), thắng bằng hàng, cột, đường chéo chính, đường chéo phụ, người 2 thắng, và ván **hòa** (đầy bàn, không ai thắng).

In [20]:
# n = 2: chỉ cần 2 quân thẳng hàng
assert check_game(2, [(0, 0, 1), (0, 1, 2), (1, 1, 1)], "n=2 chéo chính người 1")[-1] == 1
assert check_game(2, [(0, 0, 1), (1, 1, 2), (0, 1, 1)], "n=2 hàng 0 người 1")[-1] == 1
assert check_game(2, [(0, 1, 1), (0, 0, 2), (1, 0, 1)], "n=2 chéo phụ người 1")[-1] == 1
assert check_game(2, [(0, 0, 2), (1, 0, 1), (1, 1, 2)], "n=2 chéo chính người 2")[-1] == 2

# n = 3: các kiểu thắng
assert check_game(3, [(0, 0, 1), (1, 0, 2), (0, 1, 1), (1, 1, 2), (0, 2, 1)], "hàng 0 người 1")[-1] == 1
assert check_game(3, [(0, 0, 1), (0, 1, 2), (1, 0, 1), (1, 1, 2), (2, 0, 1)], "cột 0 người 1")[-1] == 1
assert check_game(3, [(0, 0, 1), (0, 1, 2), (1, 1, 1), (0, 2, 2), (2, 2, 1)], "chéo chính người 1")[-1] == 1
assert check_game(3, [(0, 2, 1), (0, 1, 2), (1, 1, 1), (0, 0, 2), (2, 0, 1)], "chéo phụ người 1")[-1] == 1
assert check_game(3, [(0, 0, 1), (1, 1, 2), (0, 1, 1), (0, 2, 2), (2, 2, 1), (2, 0, 2)], "chéo phụ người 2")[-1] == 2

# ván hòa 3x3 (X O X / X O O / O X X)
draw = [(0, 0, 1), (0, 1, 2), (0, 2, 1), (1, 1, 2), (1, 0, 1), (1, 2, 2), (2, 1, 1), (2, 0, 2), (2, 2, 1)]
ref = check_game(3, draw, "hòa 3x3")
assert ref[1:] == [0] * 9 and oracle_winner([[1, 2, 1], [1, 2, 2], [2, 1, 1]]) == 0
print("PASS: các trường hợp biên")

PASS: các trường hợp biên


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **bộ đếm không phân biệt người chơi** (cộng `+1` cho cả hai); (b) **quên đường chéo phụ** (hoặc dùng `row == col` cho cả hai đường chéo).

In [21]:
class WrongNoSign:
    def __init__(self, n):
        self.n, self.rows, self.cols, self.diag, self.anti = n, [0] * n, [0] * n, 0, 0

    def move(self, row, col, player):
        self.rows[row] += 1                      # SAI: không phân biệt người chơi
        self.cols[col] += 1
        if row == col:
            self.diag += 1
        if row + col == self.n - 1:
            self.anti += 1
        return player if self.n in (self.rows[row], self.cols[col], self.diag, self.anti) else 0


class WrongNoAnti:
    def __init__(self, n):
        self.n, self.rows, self.cols, self.diag = n, [0] * n, [0] * n, 0

    def move(self, row, col, player):
        d = 1 if player == 1 else -1
        self.rows[row] += d
        self.cols[col] += d
        if row == col:
            self.diag += d
        return player if self.n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag)) else 0     # SAI: thiếu chéo phụ


# Hàng 0 gồm X O X: người 1, 2 xen kẽ -> bản "không dấu" tưởng nhầm đã thắng
mixed = [(0, 0, 1), (0, 1, 2), (0, 2, 1)]
print("đúng:", oracle_run(3, mixed), "| không dấu:", run_commands(WrongNoSign, ["T", "move", "move", "move"], [[3]] + [[r, c, p] for r, c, p in mixed])[1:])
anti_game = [(0, 2, 1), (0, 0, 2), (1, 1, 1), (1, 0, 2), (2, 0, 1)]
print("đúng:", oracle_run(3, anti_game), "| thiếu chéo phụ:", run_commands(WrongNoAnti, ["T"] + ["move"] * 5, [[3]] + [[r, c, p] for r, c, p in anti_game])[1:])
assert run_commands(WrongNoSign, ["T", "move", "move", "move"], [[3]] + [[r, c, p] for r, c, p in mixed])[1:] != oracle_run(3, mixed)
assert run_commands(WrongNoAnti, ["T"] + ["move"] * 5, [[3]] + [[r, c, p] for r, c, p in anti_game])[1:] != oracle_run(3, anti_game)

đúng: [0, 0, 0] | không dấu: [0, 0, 1]
đúng: [0, 0, 0, 0, 1] | thiếu chéo phụ: [0, 0, 0, 0, 0]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

Sinh **ván cờ hợp lệ** theo đúng luật đề: chọn `n` từ `2` đến `8`, xáo trộn các ô, hai người **luân phiên** (người bắt đầu chọn ngẫu nhiên), **dừng ngay khi có người thắng** (không còn nước đi). Mỗi ván được chạy trên cả `8` bản cài đặt và so với oracle sau **từng nước**.

In [22]:
def random_game(rng):
    n = rng.randint(2, 8)
    cells = [(r, c) for r in range(n) for c in range(n)]
    rng.shuffle(cells)
    player = rng.choice([1, 2])
    board = [[0] * n for _ in range(n)]
    moves = []
    for r, c in cells:
        moves.append((r, c, player))
        board[r][c] = player
        if oracle_winner(board) != 0:
            break                                  # đã có người thắng: không còn nước đi
        player = 3 - player
    return n, moves


rng = random.Random(348)
wins = draws = 0
for _ in range(1500):
    n, moves = random_game(rng)
    ref = check_game(n, moves, "ngẫu nhiên")
    assert all(x == 0 for x in ref[1:-1])           # chỉ nước cuối mới có thể thắng
    if ref[-1]:
        wins += 1
    else:
        draws += 1
print("PASS: 1500 ván ngẫu nhiên;", wins, "ván có người thắng,", draws, "ván hòa")

PASS: 1500 ván ngẫu nhiên; 726 ván có người thắng, 774 ván hòa


### 12.4. Kiểm tra hiệu năng ở giới hạn lớn nhất

`n = 100` và `10⁴` nước đi **không ai thắng** (ván hòa dựng sẵn), để mọi nước đều phải chạy hết. Mẫu: ô `(r, c)` thuộc người `1 + (c + r // 2) % 2`. Mỗi hàng xen kẽ hai người, mỗi cột có chuỗi `0 0 1 1 0 0 ...` nên không đường nào đủ `100` quân một người; ta cũng kiểm bằng oracle một lần ở cuối.

In [23]:
n = 100
owner = [[1 + (c + r // 2) % 2 for c in range(n)] for r in range(n)]
assert oracle_winner(owner) == 0
cells1 = [(r, c) for r in range(n) for c in range(n) if owner[r][c] == 1]
cells2 = [(r, c) for r in range(n) for c in range(n) if owner[r][c] == 2]
assert len(cells1) == len(cells2) == n * n // 2
big_moves = []
for (r1, c1), (r2, c2) in zip(cells1, cells2):             # hai người luân phiên
    big_moves.append((r1, c1, 1))
    big_moves.append((r2, c2, 2))
assert len(big_moves) == n * n and len({(r, c) for r, c, _ in big_moves}) == n * n

ops_big = ["TicTacToe"] + ["move"] * len(big_moves)
args_big = [[n]] + [[r, c, p] for r, c, p in big_moves]
for key in ["counter", "bitmask", "leetcode", "lines"]:
    t0 = time.perf_counter()
    out = run_commands(impls[key], ops_big, args_big)
    dt = time.perf_counter() - t0
    assert out == [None] + [0] * len(big_moves), key
    print(f"{key:10s} n=100, {len(big_moves)} nước: {dt * 1000:8.1f} ms")

# Cách 1 chậm: chạy với n nhỏ hơn để minh họa chênh lệch (n = 30)
n2 = 30
owner2 = [[1 + (c + r // 2) % 2 for c in range(n2)] for r in range(n2)]
assert oracle_winner(owner2) == 0
c1 = [(r, c) for r in range(n2) for c in range(n2) if owner2[r][c] == 1]
c2 = [(r, c) for r in range(n2) for c in range(n2) if owner2[r][c] == 2]
mv = [x for pair in zip([(r, c, 1) for r, c in c1], [(r, c, 2) for r, c in c2]) for x in pair]
o2, a2 = ["TicTacToe"] + ["move"] * len(mv), [[n2]] + [[r, c, p] for r, c, p in mv]
for key in ["scan_all", "lines", "counter"]:
    t0 = time.perf_counter()
    out = run_commands(impls[key], o2, a2)
    dt = time.perf_counter() - t0
    assert out == [None] + [0] * len(mv)
    print(f"{key:10s} n=30,  {len(mv)} nước: {dt * 1000:8.1f} ms")

counter    n=100, 10000 nước:      3.5 ms
bitmask    n=100, 10000 nước:      4.3 ms
leetcode   n=100, 10000 nước:      3.8 ms
lines      n=100, 10000 nước:      8.8 ms
scan_all   n=30,  900 nước:     52.9 ms
lines      n=30,  900 nước:      0.8 ms
counter    n=30,  900 nước:      0.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tạo bàn cờ bằng `[[0] * n] * n`

In [24]:
class SaiBanCo:
    def __init__(self, n):
        self.n = n
        self.board = [[0] * n] * n                   # SAI: n hàng cùng là MỘT list

    def move(self, row, col, player):
        b, n = self.board, self.n
        b[row][col] = player
        if all(b[row][j] == player for j in range(n)) or all(b[i][col] == player for i in range(n)):
            return player
        return 0


# Người 1 đánh (0,0): vì các hàng trùng nhau, MỌI hàng đều nhận quân ở cột 0
game = SaiBanCo(2)
print(game.move(0, 0, 1), game.board)                 # 1 [[1, 0], [1, 0]]  <- báo thắng ngay nước đầu!
print("kết quả đúng của nước đầu tiên:", oracle_run(2, [(0, 0, 1)]))
assert game.board[0] is game.board[1]
assert oracle_run(2, [(0, 0, 1)]) == [0]

1 [[1, 0], [1, 0]]
kết quả đúng của nước đầu tiên: [0]


**Vì sao sai:** `[[0] * n] * n` tạo `n` tham chiếu tới **cùng một list**, nên ghi vào `board[0][0]` làm `board[1][0]` cũng đổi theo. Bàn cờ bị "nhân đôi" nước đi và có thể báo thắng sai. **Cách sửa:** `[[0] * n for _ in range(n)]`.

### Lỗi 2: Bộ đếm không phân biệt người chơi

In [25]:
game_moves = [(0, 0, 1), (0, 1, 2), (0, 2, 1)]       # hàng 0 = X O X, chưa ai thắng
print("sai (đều +1):", run_commands(WrongNoSign, ["T"] + ["move"] * 3, [[3]] + [[r, c, p] for r, c, p in game_moves]))
print("đúng        :", run_commands(TicTacToe, ["T"] + ["move"] * 3, [[3]] + [[r, c, p] for r, c, p in game_moves]))

sai (đều +1): [None, 0, 0, 1]
đúng        : [None, 0, 0, 0]


**Vì sao sai:** nếu cả hai người đều `+1` thì một hàng có `X O X` cũng đạt bộ đếm `3 = n` và bị tính là thắng. **Cách sửa:** người 1 `+1`, người 2 `−1`, và kiểm tra `abs(bộ đếm) == n`; hoặc dùng bộ đếm riêng cho từng người.

### Lỗi 3: Quên đường chéo phụ, hoặc điều kiện sai `row == n - col`

In [26]:
class SaiCheoPhu:
    def __init__(self, n):
        self.n, self.rows, self.cols, self.diag, self.anti = n, [0] * n, [0] * n, 0, 0

    def move(self, row, col, player):
        d = 1 if player == 1 else -1
        self.rows[row] += d
        self.cols[col] += d
        if row == col:
            self.diag += d
        if row + col == self.n:                        # SAI: phải là n - 1 (chỉ số bắt đầu từ 0)
            self.anti += d
        n = self.n
        return player if n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag), abs(self.anti)) else 0


ga = [(0, 2, 1), (0, 0, 2), (1, 1, 1), (1, 0, 2), (2, 0, 1)]      # người 1 thắng bằng chéo phụ
print("sai  :", run_commands(SaiCheoPhu, ["T"] + ["move"] * 5, [[3]] + [[r, c, p] for r, c, p in ga]))
print("đúng :", run_commands(TicTacToe, ["T"] + ["move"] * 5, [[3]] + [[r, c, p] for r, c, p in ga]))

sai  : [None, 0, 0, 0, 0, 0]
đúng : [None, 0, 0, 0, 0, 1]


**Vì sao sai:** với chỉ số từ `0`, đường chéo phụ là các ô có `row + col == n - 1` (ví dụ `n = 3`: `(0,2)`, `(1,1)`, `(2,0)`). Viết `== n` làm bộ đếm chéo phụ không bao giờ đủ. **Cách sửa:** `row + col == n - 1`.

### Lỗi 4: Trả về `True/False` hoặc `1` mặc định thay vì **mã người thắng**

In [27]:
class SaiKieuTraVe:
    def __init__(self, n):
        self.inner = TicTacToe(n)

    def move(self, row, col, player):
        return self.inner.move(row, col, player) != 0       # SAI: trả True/False


print("sai  :", run_commands(SaiKieuTraVe, ops, args))
print("đúng :", run_commands(TicTacToe, ops, args))
print("người 2 thắng cũng phải trả 2:", run_commands(TicTacToe, ["T", "move", "move", "move", "move", "move", "move"], [[3], [0, 0, 1], [1, 0, 2], [0, 1, 1], [1, 1, 2], [2, 2, 1], [1, 2, 2]]))

sai  : [None, False, False, False, False, False, False, True]
đúng : [None, 0, 0, 0, 0, 0, 0, 1]
người 2 thắng cũng phải trả 2: [None, 0, 0, 0, 0, 0, 2]


**Vì sao sai:** đề yêu cầu **số nguyên** `0/1/2` (`2` là người 2 thắng). Trả `True` (tức `1`) khiến cả hai người đều "là người 1 thắng" hoặc đưa ra `False` thay cho `0` (so sánh kiểu dữ liệu và với `2` đều sai). **Cách sửa:** trả đúng `player` khi thắng, `0` khi chưa.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **Design** = chọn **trạng thái nhỏ nhất đủ** để các thao tác chạy nhanh: ở đây không cần bàn cờ, chỉ cần `2n + 2` bộ đếm.
- "Một thay đổi cục bộ chỉ ảnh hưởng vài đối tượng" ⇒ **chỉ cập nhật/kiểm tra những đối tượng đó** (tối đa 4 đường qua ô vừa đánh).
- Hai bên đối kháng + đếm "ai hơn" ⇒ dùng **dấu `+1/−1`** để một bộ đếm mô tả cả hai người (mẹo cũng gặp ở bài **bỏ phiếu**, **cân bằng ngoặc**).
- Ô `(r, c)` trên đường chéo chính: `r == c`; chéo phụ: `r + c == n - 1` (**chỉ số từ `0`**).
- Luôn tự dựng **bộ chạy chuỗi lệnh** (`run_commands`) và một **oracle ngây thơ** để so sánh khi bài là bài thiết kế.
- Nếu đề thay "n quân liên tiếp" bằng "k quân liên tiếp" (như Gomoku) thì bộ đếm theo hàng/cột không còn đủ; hãy **đếm hai phía của ô vừa đánh** theo 4 hướng.

## 15. Biến thể của bài

1. **Gomoku / `k` quân liên tiếp** trên bàn `n × n` (bài tập 5): kiểm tra 4 hướng quanh ô vừa đánh.
2. **Valid Tic-Tac-Toe State (794):** cho bàn cờ, kiểm tra trạng thái có thể xảy ra trong ván thật không.
3. **Find Winner on a Tic Tac Toe Game (1275):** cho danh sách nước đi, trả `"A"`, `"B"`, `"Draw"`, `"Pending"` (bàn `3 × 3`).
4. **Bàn cờ hình chữ nhật `m × n`** với yêu cầu thắng theo hàng: bộ đếm `rows[m]`, `cols[n]`.
5. **Nhiều người chơi:** thay `±1` bằng bộ đếm riêng cho từng người (`defaultdict(Counter)`).

In [28]:
class KInRow:
    """Biến thể: người chơi thắng khi có k quân liên tiếp (hàng, cột, hoặc hai chéo) trên bàn n x n."""

    def __init__(self, n, k):
        self.n, self.k = n, k
        self.board = [[0] * n for _ in range(n)]

    def move(self, row, col, player):
        b, n = self.board, self.n
        b[row][col] = player
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):       # 4 hướng
            count = 1                                           # chính ô vừa đánh
            for sign in (1, -1):                                # đi về hai phía
                r, c = row + sign * dr, col + sign * dc
                while 0 <= r < n and 0 <= c < n and b[r][c] == player:
                    count += 1
                    r += sign * dr
                    c += sign * dc
            if count >= self.k:
                return player
        return 0


# k = n trùng với bài gốc (trên cùng ván cờ)
print(run_commands(lambda n: KInRow(n, n), ops, args), "(kỳ vọng", expected, ")")
g = KInRow(5, 4)
print([g.move(r, c, p) for r, c, p in [(2, 2, 1), (0, 0, 2), (2, 3, 1), (0, 1, 2), (2, 1, 1), (0, 2, 2), (2, 4, 1)]])   # người 1 có 2,1..2,4 -> 4 quân

[None, 0, 0, 0, 0, 0, 0, 1] (kỳ vọng [None, 0, 0, 0, 0, 0, 0, 1] )
[0, 0, 0, 0, 0, 0, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time mỗi `move` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét cả bàn | lưu bàn, kiểm tra mọi đường | `O(n²)` | `O(n²)` | baseline, oracle kiểm thử |
| 2. Quét 4 đường qua ô | chỉ đường chứa ô vừa đánh | `O(n)` | `O(n²)` | cần lưu cả bàn để vẽ/hiển thị |
| 3. Bộ đếm `+1/−1` | một số cho mỗi hàng/cột/chéo | `O(1)` | `O(n)` | **nên nộp** (follow-up) |
| 4. Bitmask | tập vị trí của mỗi người | `O(1)` | `O(n)` | cần biết **ô nào còn thiếu** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[[0] * 2 for _ in range(2)]`, `1 << 3`, `abs(-3)`, `(1 << 3) - 1`, và `run_commands(TicTacToe, ["T", "move"], [[2], [0, 0, 1]])`.

**Bài 2 (tính tay).** Bàn `n = 3`, các nước: `(0,0,1)`, `(1,1,2)`, `(0,1,1)`, `(2,2,2)`, `(0,2,1)`. Lập bảng `rows`, `cols`, `diag`, `anti` sau từng nước và cho biết kết quả từng nước.

**Bài 3 (điền TODO).** Hoàn thành lớp `TicTacToeTodo` theo Cách 3 (bộ đếm `+1/−1`).

In [29]:
class TicTacToeTodo:
    def __init__(self, n):
        # TODO 1: lưu n; tạo rows (n số 0), cols (n số 0), diag = 0, anti = 0
        pass

    def move(self, row, col, player):
        # TODO 2: delta = +1 nếu player == 1, ngược lại -1
        # TODO 3: cộng delta vào rows[row], cols[col]; thêm diag nếu row == col; thêm anti nếu row + col == n - 1
        # TODO 4: nếu abs của một trong 4 bộ đếm vừa đổi bằng n: trả về player
        # TODO 5: ngược lại trả về 0
        pass

# KHÔNG tạo đối tượng / gọi move ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết lớp `TicTacToeLines2` theo Cách 2 (lưu bàn cờ, chỉ quét **4 đường qua ô vừa đánh**) **mà không nhìn lại code**; dùng `all(...)` cho từng đường.

**Bài 5 (biến thể / mini challenge).** Viết lớp `KInRowAns(n, k)` ở mục 15 **từ đầu**: sau mỗi nước, **đếm các quân liền nhau của người vừa đánh** theo 4 hướng `(0,1)`, `(1,0)`, `(1,1)`, `(1,-1)` (đi cả hai phía) và trả `player` nếu đạt `k`. Kiểm tra với oracle quét cả bàn. Câu hỏi thêm: vì sao bộ đếm `rows/cols` của Cách 3 **không** dùng được khi `k < n`?

In [30]:
class KInRowTodo:
    def __init__(self, n, k):
        # TODO: lưu n, k; tạo bàn cờ n x n
        pass

    def move(self, row, col, player):
        # TODO: ghi quân; với mỗi hướng (dr, dc) đếm 1 + số quân liền nhau ở hai phía; trả player nếu >= k
        pass

# KHÔNG tạo đối tượng ở đây.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[[0, 0], [0, 0]]`, `8`, `3`, `7`, `[None, 0]` (bàn `n = 2` mới đánh một quân). Bài 3: bắt đầu từ `self.rows = [0] * n`; người 1 là `+1`, người 2 là `−1`. Bài 4: `board[row][col] = player` rồi `all(...)` cho hàng/cột.
- **Mức 2 (vừa):** Bài 2: nước 1: `rows=[1,0,0]`, `cols=[1,0,0]`, `diag=1`; nước 2: `rows=[1,-1,0]`, `cols=[1,-1,0]`, `diag=0`; nước 3: `rows=[2,-1,0]`, `cols=[1,0,0]`... hãy tự làm tiếp. Bài 5: mỗi hướng chạy hai vòng `while` (tiến và lùi), nhớ kiểm tra biên `0 <= r < n`.
- **Mức 3 (sâu):** Bài 2 kết quả: `[0, 0, 0, 0, 1]` (hàng 0 đủ `3` quân người 1 ở nước 5). Bài 5 (câu hỏi thêm): với `k < n`, một chuỗi `k` quân liền nhau có thể nằm **ở giữa** hàng; tổng số quân trên cả hàng không còn nói lên chúng có liền nhau không, nên phải kiểm tra **tính liền kề** quanh ô vừa đánh.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert [[0] * 2 for _ in range(2)] == [[0, 0], [0, 0]]
assert 1 << 3 == 8 and abs(-3) == 3 and (1 << 3) - 1 == 7
assert run_commands(TicTacToe, ["T", "move"], [[2], [0, 0, 1]]) == [None, 0]

# Bài 2
moves_b2 = [(0, 0, 1), (1, 1, 2), (0, 1, 1), (2, 2, 2), (0, 2, 1)]
assert run_commands(TicTacToe, ["T"] + ["move"] * 5, [[3]] + [[r, c, p] for r, c, p in moves_b2]) == [None, 0, 0, 0, 0, 1]
assert oracle_run(3, moves_b2) == [0, 0, 0, 0, 1]

# Bài 3
class TicTacToeTodoAns:
    def __init__(self, n):
        self.n = n
        self.rows = [0] * n
        self.cols = [0] * n
        self.diag = 0
        self.anti = 0

    def move(self, row, col, player):
        delta = 1 if player == 1 else -1
        self.rows[row] += delta
        self.cols[col] += delta
        if row == col:
            self.diag += delta
        if row + col == self.n - 1:
            self.anti += delta
        n = self.n
        if n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag), abs(self.anti)):
            return player
        return 0

assert run_commands(TicTacToeTodoAns, ops, args) == expected

# Bài 4
class TicTacToeLines2:
    def __init__(self, n):
        self.n = n
        self.board = [[0] * n for _ in range(n)]

    def move(self, row, col, player):
        n, b = self.n, self.board
        b[row][col] = player
        if all(b[row][j] == player for j in range(n)):
            return player
        if all(b[i][col] == player for i in range(n)):
            return player
        if row == col and all(b[i][i] == player for i in range(n)):
            return player
        if row + col == n - 1 and all(b[i][n - 1 - i] == player for i in range(n)):
            return player
        return 0

impls_ans = {"todo": TicTacToeTodoAns, "lines2": TicTacToeLines2}
rng_ans = random.Random(3480)
for _ in range(300):
    n, moves = random_game(rng_ans)
    ref = [None] + oracle_run(n, moves)
    for key, cls in impls_ans.items():
        assert run_commands(cls, ["T"] + ["move"] * len(moves), [[n]] + [[r, c, p] for r, c, p in moves]) == ref, key

# Bài 5: KInRowAns (dùng lớp KInRow ở mục 15) so với oracle quét cả bàn
def oracle_kinrow(board, k):
    n = len(board)
    for r in range(n):
        for c in range(n):
            p = board[r][c]
            if p == 0:
                continue
            for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                if all(0 <= r + i * dr < n and 0 <= c + i * dc < n and board[r + i * dr][c + i * dc] == p for i in range(k)):
                    return p
    return 0

rng5 = random.Random(3481)
for _ in range(400):
    n = rng5.randint(3, 7)
    k = rng5.randint(3, n)
    cells = [(r, c) for r in range(n) for c in range(n)]
    rng5.shuffle(cells)
    g = KInRow(n, k)
    board = [[0] * n for _ in range(n)]
    p = rng5.choice([1, 2])
    for r, c in cells:
        board[r][c] = p
        got = g.move(r, c, p)
        assert got == oracle_kinrow(board, k), (n, k, r, c)
        if got:
            break
        p = 3 - p
print("Tất cả đáp án bài tập đều pass")

Tất cả đáp án bài tập đều pass


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bộ đếm `+1/−1`)

```python
class TicTacToe:

    def __init__(self, n: int):
        self.n = n
        self.rows = [0] * n
        self.cols = [0] * n
        self.diag = 0
        self.anti = 0

    def move(self, row: int, col: int, player: int) -> int:
        delta = 1 if player == 1 else -1
        self.rows[row] += delta
        self.cols[col] += delta
        if row == col:
            self.diag += delta
        if row + col == self.n - 1:
            self.anti += delta
        n = self.n
        if n in (abs(self.rows[row]), abs(self.cols[col]), abs(self.diag), abs(self.anti)):
            return player
        return 0
```

**Checklist**

- [ ] Hai người có **dấu ngược nhau** (`+1` / `−1`) và kiểm tra bằng `abs(...) == n`?
- [ ] Cập nhật đúng **hàng `row`**, **cột `col`**, chéo chính khi `row == col`, chéo phụ khi `row + col == n - 1`?
- [ ] Trả về **`player`** (`1` hoặc `2`) khi thắng, **`0`** khi chưa — không phải `True/False`?
- [ ] Không tạo bàn cờ bằng `[[0] * n] * n` (nếu dùng Cách 1/2)?
- [ ] Đã thử `n = 2` (thắng chỉ với 2 quân) và ván hòa đầy bàn?
- [ ] Đã so với oracle quét cả bàn trên nhiều ván ngẫu nhiên đúng luật?
- [ ] Skeleton đúng `class TicTacToe` / `__init__(self, n: int)` / `move(self, row: int, col: int, player: int) -> int`?